# Convolutional Neural Networks
You should build an end-to-end machine learning pipeline using a convolutional neural network model. In particular, you should do the following:
- Load the `mnist` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Split the dataset into training and test sets using [Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
- Build an end-to-end machine learning pipeline, including a [convolutional neural network](https://keras.io/examples/vision/mnist_convnet/) model.
- Optimize your pipeline by validating your design decisions.
- Test the best pipeline on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

In [1]:
import pandas as pd
import math
import numpy as np
import tensorflow as tf
import keras
from keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/m-mahdavi/teaching/refs/heads/main/datasets/mnist.csv')
df.head()

,id,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,31953,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,34452,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,60897,5,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,36953,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1981,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Shape of original data:- {df.shape}")
print(f"Shape of train data:- {df_train.shape}")
print(f"Shape of test data:- {df_test.shape}")

Shape of original data:- (4000, 786)
Shape of train data:- (3200, 786)
Shape of test data:- (800, 786)


In [4]:
length = int(math.sqrt(df_train.shape[1]-2))

print(f"The images have {df_train.shape[1]-2} pixels.")
print(f"The images are {length}x{length} pixels.")

The images have 784 pixels.
The images are 28x28 pixels.


In [5]:
len(df_train['class'].unique())

10

In [6]:
x_train = df_train.drop(["id", "class"], axis=1)
y_train = df_train["class"]

x_test = df_test.drop(["id", "class"], axis=1)
y_test = df_test["class"]

print(f"Shape of x_train:- {x_train.shape}")
print(f"Shape of x_test:- {x_test.shape}")
print(f"Shape of y_train:- {y_train.shape}")
print(f"Shape of y_test:- {y_test.shape}")

Shape of x_train:- (3200, 784)
Shape of x_test:- (800, 784)
Shape of y_train:- (3200,)
Shape of y_test:- (800,)


In [7]:
numerical_attributes = x_train.select_dtypes(include=['int64']).columns

ct = ColumnTransformer([("scaling", MinMaxScaler(), numerical_attributes)])
ct.fit(x_train)

x_train = ct.transform(x_train)
x_test = ct.transform(x_test)

In [8]:
x_train = x_train.reshape(-1, 28, 28, 1)

In [9]:
y_train = keras.utils.to_categorical(y_train, num_classes=10)

In [10]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

In [11]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),

    layers.Dropout(0.5),

    layers.Dense(10, activation="softmax"),
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 422,154 (1.61 MB)

 Trainable params: 421,898 (1.61 MB)

 Non-trainable params: 256 (1.00 KB)

In [12]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(x_train, y_train, batch_size=128, epochs=15, validation_split=0.2, callbacks=[early_stop])

Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 226ms/step - accuracy: 0.5760 - loss: 1.3540 - val_accuracy: 0.6781 - val_loss: 1.8374
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 174ms/step - accuracy: 0.9166 - loss: 0.3069 - val_accuracy: 0.8609 - val_loss: 1.6487
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 126ms/step - accuracy: 0.9581 - loss: 0.1672 - val_accuracy: 0.9141 - val_loss: 1.5356
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 124ms/step - accuracy: 0.9750 - loss: 0.1199 - val_accuracy: 0.9422 - val_loss: 1.4447
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 132ms/step - accuracy: 0.9825 - loss: 0.0896 - val_accuracy: 0.9594 - val_loss: 1.3035
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 176ms/step - accuracy: 0.9876 - loss: 0.0718 - val_accuracy: 0.9656 - val_loss: 1.1715
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 132ms/step - accuracy: 0.9921 - loss: 0.0503 - val_accuracy: 0.9594 - val_loss: 1.0490
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 133ms/step - accuracy: 0.9907 - loss: 0.0452 - val_accuracy: 0